# Phase 9 — eksperimen konfigurasi model setelah P0

Bandingkan **dua inference aktual**, confidence 0.25 dan 0.10, pada 600 frame kontinu TGDI-WEEKDAY-SIANG. YOLO26M, ukuran input 640, BoT-SORT, ReID person-domain, dan raw tetap sama. Ini eksperimen development, bukan benchmark counting atau pengujian delapan video.


**Prasyarat:** bukti eksekusi/uji P0 tersimpan dan terikat SHA sebelum inference. Hasil baseline B v4 tetap immutable. Batas klip: konteks 7380–7979; warm-up 7380–7439; evaluasi 7440–7979. Tidak ada riwayat tracker sebelum frame 7380. Video/weights tidak dimasukkan ke GitHub.


In [ ]:
from pathlib import Path
import sys, hashlib, json
REPO_ROOT = Path(globals().get("REPO_ROOT", "/kaggle/working/traffic_project"))
sys.path.insert(0, str(REPO_ROOT / "notebooks"))
from phase_9_model_experiment import make_plan, validate_plan, run_experiment
VIDEO_PATH = Path(globals().get("VIDEO_PATH", "/kaggle/input/datasets/chrisbiran/tgdi-traffic-videos/TGDI-WEEKDAY-SIANG.mp4"))
P0_GATE_PATH = REPO_ROOT / "results/b_v5/verification/p0_execution_gate.json"
PROFILE_PATH = REPO_ROOT / "docs/evidence/phase_8/config/processing_profile_v1.json"
OUTPUT_PATH = Path("/kaggle/working/phase9_model_confidence_v1")
RUN_EXPERIMENT = False


In [ ]:
if not P0_GATE_PATH.is_file():
    raise RuntimeError("Bukti P0 belum tersedia; jangan menjalankan inference model dahulu")
gate = json.loads(P0_GATE_PATH.read_text())
if gate.get("status") != "PASS":
    raise RuntimeError("Gate P0 belum PASS; periksa bukti dan batasnya")
plan = make_plan(PROFILE_PATH, p0_gate_sha256=hashlib.sha256(P0_GATE_PATH.read_bytes()).hexdigest(), p0_gate_path=P0_GATE_PATH,
                 video_path=VIDEO_PATH, output_path=OUTPUT_PATH)
validate_plan(plan)
print(json.dumps(plan, indent=2))


In [ ]:
if RUN_EXPERIMENT:
    report = run_experiment(plan)
    print(json.dumps({"status": report["status"], "variants": {k:v["counts"] for k,v in report["variants"].items()}}, indent=2))
else:
    print("PREPARED_ONLY — RUN_EXPERIMENT=False; tidak ada inference")


Review sumber dan trajectories fisik pada kedua varian. Jangan menafsirkan tambahan deteksi, ID, atau confidence sebagai kendaraan tambahan atau peningkatan akurasi. Periksa false track, identity switch, fragmentasi, objek parkir, serta unit rider/motor. Counting P0, bila direplay pada cache baru oleh notebook terpisah, harus memakai aturan yang sama dan mencatat batas konteks klip.
